# Phase 1 — Load, Explore & Clean
**Dataset:** Ames Housing (2,930 records, 82 columns)  
**Goal:** Load the raw data, understand its structure, fix all quality issues, and save a clean file ready for feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the raw dataset using a relative path
df = pd.read_csv('data/raw/AmesHousing.csv')

print('Dataset loaded successfully!')
df.head()

## Step 1 — Shape & Basic Info
First we check how big the dataset is and confirm it loaded correctly.

In [ ]:
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
df.info()

## Step 2 — Data Type Fixes
`.info()` reveals columns stored as the wrong type. We fix at least 2:
- `MS SubClass` is an integer but represents a **categorical** house class — it should be `object`.
- `Mo Sold` is an integer but represents a **month category** — converting to string prevents the model treating Dec (12) as numerically greater than Jan (1).

In [ ]:
print('Before fixes:')
print(df[['MS SubClass', 'Mo Sold']].dtypes)

# Fix 1: MS SubClass encodes building class — treat as category, not a number
df['MS SubClass'] = df['MS SubClass'].astype(str)

# Fix 2: Mo Sold is a month label (1–12), not a continuous quantity
df['Mo Sold'] = df['Mo Sold'].astype(str)

print('\nAfter fixes:')
print(df[['MS SubClass', 'Mo Sold']].dtypes)

## Step 3 — Missing Values
We identify which columns have missing data and decide how to handle each one.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df)

## Step 4 — Outlier Detection (Boxplot + IQR)
We visualise `SalePrice` with a boxplot and apply the IQR method to identify extreme values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of SalePrice before capping
axes[0].boxplot(df['SalePrice'].dropna())
axes[0].set_title('SalePrice — Before Outlier Treatment')
axes[0].set_ylabel('Price ($)')
axes[0].set_xlabel('SalePrice')

# IQR method to show the outlier boundary
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1
upper_iqr = Q3 + 1.5 * IQR
n_outliers = (df['SalePrice'] > upper_iqr).sum()

axes[1].hist(df['SalePrice'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(upper_iqr, color='red', linestyle='--', label=f'IQR upper fence: ${upper_iqr:,.0f}')
axes[1].axvline(df['SalePrice'].quantile(0.99), color='orange', linestyle='--',
                label=f'99th percentile: ${df["SalePrice"].quantile(0.99):,.0f}')
axes[1].set_title('SalePrice Distribution with Outlier Boundaries')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'IQR upper fence: ${upper_iqr:,.0f}')
print(f'Number of outliers above IQR fence: {n_outliers}')
print('Decision: Cap at the 99th percentile to retain data while reducing extreme skew.')

## Step 5 — The `clean_data()` Function
All cleaning steps are wrapped into a single reusable function. This means we can call it on any new batch of data and get the same result.

In [ ]:
def clean_data(data_frame):
    """
    Full cleaning pipeline for the Ames Housing dataset.
    Steps: fix types → remove duplicates → drop high-missing cols
           → impute remaining → cap outliers.
    Returns a clean copy of the DataFrame.
    """
    df_cleaned = data_frame.copy()

    # --- Fix data types ---
    # MS SubClass codes a building class, not a continuous number
    df_cleaned['MS SubClass'] = df_cleaned['MS SubClass'].astype(str)
    # Mo Sold is a month label, not a magnitude
    df_cleaned['Mo Sold'] = df_cleaned['Mo Sold'].astype(str)

    # --- Remove duplicate rows ---
    before = len(df_cleaned)
    df_cleaned.drop_duplicates(inplace=True)
    print(f'Duplicates removed: {before - len(df_cleaned)}')

    # --- Drop columns with >80% missing: not enough signal to impute ---
    # Pool QC (99.6% missing), Misc Feature (96.4%), Alley (93.2%), Fence (80.4%)
    cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
    df_cleaned.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # --- Impute remaining numerical missing values with median ---
    # Median is robust to the skewness present in area/price columns
    numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_cleaned[col].isnull().any():
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

    # --- Impute remaining categorical missing values with mode ---
    # Mode fills the most common category, preserving the original distribution
    cat_cols = df_cleaned.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df_cleaned[col].isnull().any():
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

    # --- Cap SalePrice outliers at the 99th percentile ---
    # Extreme values distort later regression — capping retains the records
    upper_limit = df_cleaned['SalePrice'].quantile(0.99)
    df_cleaned['SalePrice'] = np.where(
        df_cleaned['SalePrice'] > upper_limit, upper_limit, df_cleaned['SalePrice']
    )
    print(f'SalePrice capped at: ${upper_limit:,.0f}')

    return df_cleaned


# Apply the function to the raw data
df_final = clean_data(df)
print(f'\nFinal shape: {df_final.shape}')

## Step 6 — Post-Cleaning Quality Checks
Three assertions verify the data is in the expected state before we proceed.

In [ ]:
print('=== Post-Cleaning Quality Checks ===')

# Check 1: No null values remain in any column
assert df_final.isnull().any().any() == False, 'FAIL: There are still null values!'
print('Check 1 PASSED — No null values remain.')

# Check 2: All SalePrice values are positive (no zero or negative prices)
assert (df_final['SalePrice'] > 0).all(), 'FAIL: Some SalePrice values are <= 0!'
print('Check 2 PASSED — All SalePrice values are positive.')

# Check 3: The dataset has the expected number of columns (78 after dropping 4)
expected_cols = 78
assert df_final.shape[1] == expected_cols, f'FAIL: Expected {expected_cols} columns, got {df_final.shape[1]}!'
print(f'Check 3 PASSED — Column count is {df_final.shape[1]} as expected.')

print('\nAll checks passed! The data is clean and ready for Phase 2.')

In [ ]:
import os
os.makedirs('data/cleaned', exist_ok=True)
df_final.to_csv('data/cleaned/Ames_Housing_Cleaned.csv', index=False)
print('Cleaned data saved to data/cleaned/Ames_Housing_Cleaned.csv')